# Setup env

In [1]:
import pandas as pd
from pathlib import Path

In [6]:
#  Constants
data_path = '../../data/bronze/imdb_raw_data/'

# Get the absolute path of where you are right now
src_path = Path.cwd()
project_src = src_path.parents[1]
print(f"Your project source root is: {project_src}")

Your project source root is: c:\Users\davib\Desktop\MSc_DataScience\DataWarehouse\lab_assignment\imbd


In [ ]:
# Read raw data
title_basics = pd.read_csv(f'{data_path}/title.basics.tsv/data.tsv', sep='\t', na_values='\\N')
title_basics = title_basics[title_basics['titleType'].isin(['movie', 'tvSeries'])]
name_basics = pd.read_csv(f'{data_path}name.basics.tsv/data.tsv', sep='\t', dtype={'birthYear': 'Int64', 'deathYear': 'Int64'}, na_values='\\N')
title_principals = pd.read_csv(f'{data_path}/title.principals.tsv/data.tsv', sep='\t', na_values='\\N')

# Dimensional tables

## dim_profession | bridge_profession_group

In [4]:
# # Create dim_profession from unique professions in primaryProfession
# # Extract all unique professions from comma-separated primaryProfession values

# # Sources:
# # name_basics

# # 2. Create dim_profession (Vectorized unique extraction)
# unique_profs = (
#     name_basics['primaryProfession']
#     .str.split(',')
#     .explode()
#     .dropna()
#     .unique()
# )

# dim_profession = pd.DataFrame({'profession_nm': sorted(unique_profs)})
# dim_profession['profession_id'] = [f'prf_{i+1}' for i in range(len(dim_profession))]


# # 3. Create bridge_profession_group (Vectorized expansion)
# # Explode the professions column into individual rows
# bridge_profession_group = (
#     name_basics[['nconst', 'primaryProfession']]
#     .dropna(subset=['primaryProfession'])
#     .assign(profession_nm=lambda x: x['primaryProfession'].str.split(','))
#     .explode('profession_nm')
# )

# # 4. Map the IDs using Merge (Replaces the dictionary lookup loop)
# bridge_profession_group = (
#     bridge_profession_group
#     .reset_index()
#     .merge(dim_profession.reset_index(), on='profession_nm')
#     [['nconst', 'profession_id']]
# )

# # Create profession_group_id for each unique combination of professions per person
# bridge_profession_group['profession_group_id'] = 'p_grp_' + (bridge_profession_group.groupby('nconst').ngroup() + 1).astype(str)
# bridge_profession_group['weighting_factor_prf'] = 1 / bridge_profession_group.groupby('nconst')['profession_id'].transform('count')


# # Write parquet files
# bridge_profession_group.to_parquet(f'{project_src}\\data\\silver\\bridge_profession_group.parquet', compression='snappy', engine='pyarrow')
# dim_profession.to_parquet(f'{project_src}\\data\\silver\\dim_profession.parquet', compression='snappy', engine='pyarrow')

# # del bridge_profession_group
# # del dim_profession
# # print(f"\nbridge_profession_group shape: {bridge_profession_group.shape}")
# # print(f"Sample bridge data:\n{bridge_profession_group.head(10)}")

## dim_person

In [5]:
# # Create dim_person from name_basics
# # Load name_basics data

# # Sources:
# # name_basics

# # Select required columns for dim_person
# dim_person = name_basics[['nconst', 'primaryName', 'birthYear', 'deathYear']].copy()

# # Join with bridge_profssion_group to get profession_group_id
# per_prfession = dim_person.merge(bridge_profession_group, on='nconst', how='left')

# # Remove duplicates if any and set nconst as index
# dim_person = per_prfession.drop_duplicates(subset=['nconst'])
# dim_person = dim_person[['nconst','primaryName', 'birthYear', 'deathYear', 'profession_group_id', 'weighting_factor_prf']]

# print(f"dim_person shape: {dim_person.shape}")
# # Write parquet files
# dim_person.to_parquet(f'{project_src}\\data\\silver\\dim_person.parquet', compression='snappy', engine='pyarrow')

# del dim_person

## dim_roles

In [6]:
# # Create dim_roles from title_principals and title_basics
# # Load title_principals and title_basics data

# # Sources:
# # title_principals
# # title_basics

# # Join title_principals with title_basics to get titleType
# dim_roles = title_principals.merge(title_basics[['tconst', 'titleType']], on='tconst', how='left')

# # Create a unique role_id combining nconst, tconst, and ordering
# dim_roles['role_id'] = dim_roles['nconst'] + '_' + dim_roles['tconst'] + '_' + dim_roles['ordering'].astype(str)

# # Select and reorder columns
# dim_roles = dim_roles[['role_id', 'nconst', 'tconst', 'titleType', 'category', 'job', 'characters']].copy()

# # Remove duplicates and set role_id as index
# dim_roles = dim_roles.drop_duplicates(subset=['role_id'])
# print(f"dim_roles shape: {dim_roles.shape}")

# # Write parquet files
# dim_roles.to_parquet(f'{project_src}\\data\\silver\\dim_roles.parquet', compression='snappy', engine='pyarrow')

# del dim_roles

## dim_genre | bridge_genre_group

In [9]:
# 1. Load Data
# Sources:
# title_basics

# 2. Create dim_genres (The Dimension Table)
# Extract unique genres using vectorized split and unique()
unique_genres = (
    title_basics['genres']
    .str.split(',')
    .explode()
    .dropna()
    .unique()
)

dim_genres = pd.DataFrame({'genre_nm': sorted(unique_genres)})
dim_genres['genre_id'] = [f'gen_{i+1}' for i in range(len(dim_genres))]

# 3. Create bridge_genres (The Bridge Table)
# Instead of a dictionary lookup, we use explode and then merge with dim_genres
bridge_genres = (
    title_basics[['tconst', 'genres']]
    .dropna(subset=['genres'])
    .assign(genre_nm=lambda x: x['genres'].str.split(','))
    .explode('genre_nm')
)

# Merge to get the genre_id (vectorized replacement for the dictionary lookup)
bridge_genres = (
    bridge_genres.reset_index()
    .merge(dim_genres.reset_index(), on='genre_nm')
    [['tconst', 'genre_id']]
)

# 4. Create genre_group_id
# This stays vectorized as before
bridge_genres['genre_group_id'] = (
    'g_grp_' + 
    (bridge_genres.groupby('tconst').ngroup() + 1).astype(str)
)
# Calculate the weight as 1 divided by the number of genres for that specific title
bridge_genres['weighting_factor_gen'] = 1 / bridge_genres.groupby('tconst')['genre_id'].transform('count')

print(f"dim_genres shape: {dim_genres.shape}")
print(f"bridge_genres shape: {bridge_genres.shape}")

# Write parquet files
dim_genres.to_parquet(f'{project_src}\\data\\silver\\dim_genres.parquet', compression='snappy', engine='pyarrow')
bridge_genres.to_parquet(f'{project_src}\\data\\silver\\bridge_genres.parquet', compression='snappy', engine='pyarrow')

# del dim_genres
# del bridge_genres

dim_genres shape: (28, 2)
bridge_genres shape: (1022855, 4)


## dim_title_basic 

In [11]:
# Sources:
# title_basics

# Select required columns for dim_title_basic
dim_title_basic = title_basics[['tconst', 'titleType','primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'runtimeMinutes']].copy()

# Join with bridge_profssion_group to get profession_group_id
title_genre = dim_title_basic.join(bridge_genres.set_index('tconst'), on='tconst', how='left')
title_genre = title_genre[['tconst', 'titleType','primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'runtimeMinutes', 'genre_group_id']].copy()
# Remove duplicates if any and set tconst as index
dim_title_basic = title_genre.drop_duplicates(subset=['tconst'])

# Convert to numeric, forcing errors to NaN
dim_title_basic['runtimeMinutes'] = pd.to_numeric(dim_title_basic['runtimeMinutes'], errors='coerce')

# dim_title_basic = dim_title_basic[['tconst', 'titleType','primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'runtimeMinutes', 'genre_group_id']]
print(f"dim_title_basic shape: {dim_title_basic.shape}")

#Write parquet files
dim_title_basic.to_parquet(f'{project_src}\\data\\silver\\dim_title_basic.parquet', compression='snappy', engine='pyarrow')

del dim_title_basic

dim_title_basic shape: (773862, 9)


##  bridge_kt_group

In [12]:
# 1. Load data (Keeping your existing logic)
# Note: Using na_values='\\N' as discussed previously is perfect here

# Sources:
# title_basics
# name_basics 

# 2. Vectorized Expansion (Replacing the two loops)
# We select only the columns we need, split the strings into lists, and "explode" them
bridge_kwn_titles = (
    name_basics[['nconst', 'knownForTitles']]
    .dropna(subset=['knownForTitles'])
    .assign(tconst=lambda x: x['knownForTitles'].str.split(','))
    .explode('tconst')
    [['tconst', 'nconst']]  # Reorder to match your desired output
)

# 3. Vectorized ID Generation
# ngroup() is already vectorized, so we just keep this logic
bridge_kwn_titles['kwn_title_group_id'] = (
    'kwn_t_grp_' + 
    (bridge_kwn_titles.groupby('tconst').ngroup() + 1).astype(str)
)
bridge_kwn_titles['weighting_factor_grp'] = 1 / bridge_kwn_titles.groupby('nconst')['tconst'].transform('count')


# 4. Get Unique Titles Set (If still needed for other logic)
known_titles = set(bridge_kwn_titles['tconst'].unique())

print(f"bridge_kwn_titles shape: {bridge_kwn_titles.shape}")
bridge_kwn_titles.to_parquet(f'{project_src}\\data\\silver\\bridge_kwn_titles.parquet', compression='snappy', engine='pyarrow')

del bridge_kwn_titles

bridge_kwn_titles shape: (16851455, 4)


# Factual tables

## participations

In [ ]:
# # Read dims
# dim_person = pd.read_parquet(f'{project_src}\\data\\silver\\dim_person.parquet')
# dim_roles = pd.read_parquet(f'{project_src}\\data\\silver\\dim_roles.parquet')
# dim_title_basic = pd.read_parquet(f'{project_src}\\data\\silver\\dim_title_basic.parquet')
# bridge_kwn_titles = pd.read_parquet(f'{project_src}\\data\\silver\\bridge_kwn_titles.parquet')


# participations_pers = (
#     dim_title_basic[['tconst','titleType', 'primaryTitle', 'genre_group_id']]
#     .merge(dim_roles[['tconst', 'nconst', 'category', 'job', 'characters']],on='tconst', how='left')
#     .merge(dim_person[['nconst', 'primaryName',  'profession_group_id']], on='nconst', how='left')
#     .merge(bridge_kwn_titles, on=['tconst', 'nconst'], how='left')
# )
# participations_pers = participations_pers[['nconst', 'primaryName', 'tconst', 'titleType', 'primaryTitle', 'genre_group_id', 'category', 'job', 'characters', 'profession_group_id', 'kwn_title_group_id']].copy()
# participations_pers[participations_pers['primaryName'] == 'Brad Pitt']

# print(f"participations_pers.shape: {participations_pers.shape}")

# #Write parquet file
# participations_pers.to_parquet(f'{project_src}\\data\\silver\\participations_pers.parquet', compression='snappy', engine='pyarrow')

# del participations_pers

participations_pers.shape: (5231306, 11)


# Write to silver